Modelo baseado em ilhas.
- Topologia Anel


Técnicas de Implementação
- Técnica de Migração migRing(Framework DEAP)
Programação concorrente
- SCOOP:(http://scoop.readthedocs.io/en/0.7

In [38]:
!pip install deap
!pip install scoop

In [39]:
import array
import random
import numpy

from deap import algorithms
from deap import base
from deap import creator
from deap import tools
#Biblioteca execução paralelismo
from scoop import futures

In [40]:
def fitnessFunction(chromosome):
  return sum(chromosome),

In [57]:
# Cria o tipo de função fitness e indivíduo
creator.create("Maximization", base.Fitness, weights=(1.0,))
creator.create("Genes", list, fitness=creator.Maximization)

toolbox = base.Toolbox()

# Registra os nomes e osa tipos de indivíduo, fitness e população
toolbox.register("Atributo", random.randint, 0, 1)
toolbox.register("Cromossomo", tools.initRepeat, creator.Genes, toolbox.Atributo, 300) # 50 bits de 0 e 1
toolbox.register("Populacao", tools.initRepeat, list, toolbox.Cromossomo) # é uma lista de listas que são cromossomos

# Registra os operadores. Deve-se manter os nomes evaluate, mate, mutate e select.
toolbox.register("evaluate", fitnessFunction)
toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.01)
toolbox.register("select", tools.selTournament, tournsize=5)

# Registra a função de migração
toolbox.register("migRing", tools.migRing)

In [58]:
# Registra a execução em paralelismo
toolbox.register("map", futures.map)

In [72]:
# Parâmetros do Algoritmo
prob_cx = 0.8
prob_mt = 0.1
nger = 400
freq = 10 # de 10 em 10 as poplações vão trocar indivíduos entre si
tamPop = 30
nIlhas = 5

In [73]:
# Faz a execução do AG em Ilhas
islands = [toolbox.Populacao(n=tamPop) for _ in range(nIlhas)]
toolbox.register("algorithm", algorithms.eaSimple, toolbox=toolbox, cxpb=prob_cx, mutpb=prob_mt, ngen=freq, verbose=False)
for _ in range(0, nger, freq):
  results = toolbox.map(toolbox.algorithm, islands)
  islands = [pop for pop, logbook in results]
  toolbox.migRing(islands, 5, tools.selBest)

In [74]:
# Imprime o fitness máximo e médio por Ilhas
for pop in islands:
  # Armazena fitness final de cada ilha.
  finalFitness = []
  for individual in pop:
    finalFitness.append(individual.fitness.values)
  print(f'Max Fitness: {max(finalFitness)} - Avg Fitness: {numpy.mean(finalFitness)}')


Max Fitness: (300.0,) - Avg Fitness: 300.0
Max Fitness: (300.0,) - Avg Fitness: 299.8666666666667
Max Fitness: (300.0,) - Avg Fitness: 299.6666666666667
Max Fitness: (300.0,) - Avg Fitness: 299.3666666666667
Max Fitness: (300.0,) - Avg Fitness: 299.8
